In [1]:
!pip install jiwer -q

import nltk
nltk.download('punkt', quiet=True)

True

In [2]:
#IMPORTS
import os
import re
import cv2
import time
import math
import torch
import pickle
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
 
from jiwer import wer
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
#Paths
train_i3d_path      = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/i3d_features_rwth phoenix 2014t/i3d_features_rwth phoenix 2014t/train"
train_mediapipe_path= "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/mediapipe_features_rwth phoenix weather 2014t/mediapipe_features/train"
val_i3d_path        = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/i3d_features_rwth phoenix 2014t/i3d_features_rwth phoenix 2014t/val"
val_mediapipe_path  = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/mediapipe_features_rwth phoenix weather 2014t/mediapipe_features/val"
train_tsv_path      = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/tsv files_rwth phoenix 2014t/tsv files/cvpr23.fairseq.i3d.train.how2sign.tsv"
val_tsv_path        = "/kaggle/input/datasets/vedijaiswal/cslrid3-mediapipe/tsv files_rwth phoenix 2014t/tsv files/cvpr23.fairseq.i3d.val.how2sign.tsv"
 

In [4]:
#DATASET
class CSLRDataset(Dataset):
 
    def __init__(self, i3d_path, mediapipe_path, df,
                 text_col="translation", target_len=128, augment=False):
 
        self.i3d_path       = i3d_path
        self.mediapipe_path = mediapipe_path
        self.target_len     = target_len
        self.augment        = augment
 
        self.label_map = dict(zip(df["id"], df[text_col]))
 
        i3d_files    = set(os.listdir(i3d_path))
        mp_files     = set(os.listdir(mediapipe_path))
        common_files = sorted(list(i3d_files & mp_files))
 
        self.files = [
            f for f in common_files
            if os.path.splitext(f)[0] in self.label_map
        ]
        print(f"Matched common files: {len(self.files)}")
 
    def __len__(self):
        return len(self.files)
 
    def temporal_resize(self, features):
        return cv2.resize(
            features,
            (features.shape[1], self.target_len),
            interpolation=cv2.INTER_LINEAR
        )
 
    # ── IMPROVEMENT 1: Stronger augmentation ──────────────
    def augment_features(self, x):
 
        # (a) Gaussian noise — slightly stronger than before (0.015 vs 0.01)
        if np.random.rand() < 0.8:
            noise = np.random.normal(0, 0.015, x.shape)
            x = x + noise
 
        # (b) TIME STRETCH — most impactful new addition
        #     Simulates different signing speeds (0.75x to 1.25x)
        if np.random.rand() < 0.5:
            factor  = np.random.uniform(0.75, 1.25)
            new_T   = max(10, int(x.shape[0] * factor))
            indices = np.linspace(0, x.shape[0] - 1, new_T).astype(int)
            x       = x[indices]
            x       = self.temporal_resize(x)   # back to TARGET_LEN
 
        # (c) FEATURE MASKING — randomly zero out 10% of feature dims
        #     Teaches model to be robust to missing keypoints
        if np.random.rand() < 0.3:
            mask_cols = int(x.shape[1] * 0.10)
            start     = np.random.randint(0, x.shape[1] - mask_cols)
            x[:, start:start + mask_cols] = 0
 
        # (d) TEMPORAL MASKING — zero out a random time segment
        #     Like SpecAugment but for video features
        if np.random.rand() < 0.3:
            t_start = np.random.randint(0, max(1, x.shape[0] - 15))
            t_end   = min(x.shape[0], t_start + np.random.randint(5, 15))
            x[t_start:t_end, :] = 0
 
        # (e) Random frame drop (kept from original)
        if np.random.rand() < 0.3:
            keep_ratio = np.random.uniform(0.85, 0.95)
            keep_len   = max(10, int(len(x) * keep_ratio))
            idx        = np.sort(np.random.choice(len(x), keep_len, replace=False))
            x          = x[idx]
            x          = self.temporal_resize(x)
 
        return x
 
    def __getitem__(self, idx):
        fname = self.files[idx]
 
        i3d = np.load(os.path.join(self.i3d_path, fname))
        mp  = np.load(os.path.join(self.mediapipe_path, fname))
        mp  = mp.reshape(mp.shape[0], -1)
 
        i3d = self.temporal_resize(i3d)
        mp  = self.temporal_resize(mp)
 
        x = np.concatenate([i3d, mp], axis=1)
 
        if self.augment:
            x = self.augment_features(x)
 
        x          = torch.tensor(x, dtype=torch.float32)
        video_id   = os.path.splitext(fname)[0]
        sentence   = self.label_map[video_id]
 
        return x, sentence
 


In [5]:
#COLLATE
def collate_fn(batch):
    features  = [item[0] for item in batch]
    sentences = [item[1] for item in batch]
    features  = pad_sequence(features, batch_first=True)
    return features, sentences

In [6]:
#VOCABULARY
train_df = pd.read_csv(train_tsv_path, sep="\t")
val_df   = pd.read_csv(val_tsv_path,   sep="\t")
 
vocab = set()
for sentence in train_df["translation"]:
    vocab.update(sentence.lower().split())
 
vocab    = ["<blank>", "<sos>", "<eos>"] + sorted(list(vocab))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
 
SOS_IDX   = word2idx["<sos>"]
EOS_IDX   = word2idx["<eos>"]
BLANK_IDX = word2idx["<blank>"]
 
print("Vocabulary size:", len(vocab))
print("SOS:", SOS_IDX, "| EOS:", EOS_IDX)

Vocabulary size: 2890
SOS: 1 | EOS: 2


In [7]:
#DATASETS & LOADERS
train_dataset = CSLRDataset(
    i3d_path=train_i3d_path,
    mediapipe_path=train_mediapipe_path,
    df=train_df,
    text_col="translation",
    augment=True                        # augmentation ON for training
)
 
val_dataset = CSLRDataset(
    i3d_path=val_i3d_path,
    mediapipe_path=val_mediapipe_path,
    df=val_df,
    text_col="translation",
    augment=False                       # NO augmentation for validation
)
 
loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
 
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
 
print(f"Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples")
 

Matched common files: 7096
Matched common files: 519
Train: 7096 samples | Val: 519 samples


In [8]:
#MODEL
class MultiScaleCNN(nn.Module):
 
    def __init__(self, channels):
        super().__init__()
        self.branch3 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.branch5 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.branch7 = nn.Conv1d(channels, channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm1d(channels * 3)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
 
    def forward(self, x):
        x = torch.cat([self.branch3(x), self.branch5(x), self.branch7(x)], dim=1)
        return self.dropout(self.relu(self.bn(x)))
 
 
class CSLRModel(nn.Module):
 
    def __init__(self, vocab_size, d_model=256, nhead=8,
                 num_decoder_layers=4, max_len=128):
        super().__init__()
        self.d_model = d_model
 
        # Encoder
        self.proj = nn.Linear(1123, 512)
        self.conv = MultiScaleCNN(512)
        self.lstm = nn.LSTM(1536, d_model, num_layers=2,
                                    batch_first=True, bidirectional=True, dropout=0.3)
        self.encoder_proj = nn.Linear(d_model * 2, d_model)
        self.encoder_norm = nn.LayerNorm(d_model)
 
        # Decoder
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        self.pos_encoding = nn.Embedding(max_len + 2, d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead,dim_feedforward=1024, dropout=0.3, batch_first=True)
        
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        
        self.fc_out = nn.Linear(d_model, vocab_size)
 
    def encode(self, x):
        x = self.proj(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = self.encoder_norm(self.encoder_proj(x))
        return x
 
    def decode(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        B, S = tgt.shape
        pos = torch.arange(S, device=tgt.device).unsqueeze(0).expand(B, -1)
        tgt_emb = self.embedding(tgt) + self.pos_encoding(pos)
        out = self.decoder(tgt=tgt_emb, memory=memory,tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(out)
 
    def forward(self, x, tgt):
        memory = self.encode(x)
        
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1]).to(x.device)
        return self.decode(tgt, memory, tgt_mask=tgt_mask)
 
 
model = CSLRModel(
    vocab_size=len(vocab),
    d_model=256,
    nhead=8,
    num_decoder_layers=4,
    max_len=128
).to(device)
 
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
 


Model parameters: 15,624,778


In [9]:
# LOAD TRAINED WEIGHTS

model.load_state_dict(
    torch.load(
        "/kaggle/input/models/vedijaiswal/model-best-aug/pytorch/default/1/augseq2seq_best (3).pth",
        map_location=device
    )
)

model.eval()

print("✅ Trained weights loaded")

✅ Trained weights loaded


In [10]:
# CELL 12: Improved beam search 
# Changes vs original:
#   (a) Repetition penalty — discourages repeating recent tokens
#   (b) Proper EOS handling — only truly-completed beams win
#   (c) Alpha=0.8 — rewards longer sequences slightly more
def beam_search_generate(model, features, idx2word, beam_width=10, max_len=50, alpha=0.8):
    model.eval()
    features = features.unsqueeze(0).to(device)
 
    with torch.no_grad():
        memory = model.encode(features)
        beams = [([SOS_IDX], 0.0)]
        completed = []
 
        for _ in range(max_len):
            new_beams = []
 
            for tokens, score in beams:
                if tokens[-1] == EOS_IDX:
                    completed.append((tokens, score))
                    continue
 
                tgt = torch.tensor([tokens], dtype=torch.long).to(device)
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1]).to(device)
                
                output = model.decode(tgt, memory, tgt_mask=tgt_mask)
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                topk  = torch.topk(probs, beam_width)
 
                recent_tokens = set(tokens[-10:])   # last 10 tokens
 
                for i in range(beam_width):
                    next_token = topk.indices[0][i].item()
                    next_score = score + topk.values[0][i].item()

                    # prevent EOS too early
                    if next_token == EOS_IDX and len(tokens) < 6:
                        next_score -= 10.0
 
                    # (a) REPETITION PENALTY
                    if next_token in recent_tokens:
                        next_score -= 0.6
 
                    new_beams.append((tokens + [next_token], next_score))
 
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
 
        completed.extend(beams)
 
        # (b) prefer beams that actually hit EOS
        truly_done = [b for b in completed if b[0][-1] == EOS_IDX]
        pool = truly_done if truly_done else completed
 
        # (c) length-normalised score with alpha=0.8
        best_tokens = sorted(
            pool,
            key=lambda x: x[1] / (len(x[0]) ** alpha),
            reverse=True
        )[0][0]
 
        words = [
            idx2word[idx] for idx in best_tokens
            if idx not in (SOS_IDX, EOS_IDX, BLANK_IDX) and idx in idx2word
        ]
        return " ".join(words)
 

In [11]:
# CELL 13: Improved post-processing 
def fix_common_errors(text):
    words = text.split()
    cleaned = []
    for w in words:
        if cleaned and cleaned[-1] == w:
            continue
        cleaned.append(w)
    text = " ".join(cleaned)
    # remove repeated 2-word and 3-word phrases
    text = re.sub(r'\b(\w+ \w+)( \1)+\b', r'\1', text)
    text = re.sub(r'\b(\w+ \w+ \w+)( \1)+\b', r'\1', text)
    return text

In [12]:
# GREEDY WER (fast baseline)
from jiwer import wer as compute_wer

refs_g, hyps_g = [], []
with torch.no_grad():
    for features, sentences in val_loader:
        features = features.to(device)
        for i in range(features.shape[0]):
            generated = [SOS_IDX]
            memory = model.encode(features[i].unsqueeze(0))
            for _ in range(50):
                tgt = torch.tensor([generated], dtype=torch.long).to(device)
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(generated)).to(device)
                output = model.decode(tgt, memory, tgt_mask=tgt_mask)
                next_token = output[:, -1, :].argmax(-1).item()
                if next_token == EOS_IDX:
                    break
                generated.append(next_token)
            pred = " ".join([idx2word[i] for i in generated[1:] if i in idx2word])
            refs_g.append(sentences[i].lower())
            hyps_g.append(pred.lower())

print(f"GREEDY WER: {compute_wer(refs_g, hyps_g):.4f}")

GREEDY WER: 0.8021


In [13]:
#  CELL 14: Final beam WER 
refs_b, hyps_b = [], []
 
with torch.no_grad():
    for features, sentences in val_loader:
        features = features.to(device)
        for i in range(features.shape[0]):
            pred = beam_search_generate(model, features[i], idx2word, beam_width=10)
            pred = fix_common_errors(pred)
            refs_b.append(sentences[i].lower())
            hyps_b.append(pred.lower())
 
beam_wer = wer(refs_b, hyps_b)
print(f"\nFINAL BEAM SEARCH WER: {beam_wer:.4f}")



FINAL BEAM SEARCH WER: 0.7504


In [14]:
# CELL 15: BLEU scores 
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
 
smoothie = SmoothingFunction().method1
refs_nl  = [[r.split()] for r in refs_b]
hyps_nl  = [h.split()   for h in hyps_b]
 
b1 = corpus_bleu(refs_nl, hyps_nl, weights=(1,0,0,0),smoothing_function=smoothie)
b2 = corpus_bleu(refs_nl, hyps_nl, weights=(0.5,0.5,0,0),smoothing_function=smoothie)
b3 = corpus_bleu(refs_nl, hyps_nl, weights=(0.33,0.33,0.33,0),smoothing_function=smoothie)
b4 = corpus_bleu(refs_nl, hyps_nl, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)
 
print(f"BLEU-1: {b1:.4f}")
print(f"BLEU-2: {b2:.4f}")
print(f"BLEU-3: {b3:.4f}")
print(f"BLEU-4: {b4:.4f}")


BLEU-1: 0.3747
BLEU-2: 0.2708
BLEU-3: 0.2102
BLEU-4: 0.1652


In [15]:
# =========================================
# RANDOM PREDICTION TEST
# =========================================

import random
import torch
import torch.nn.functional as F
import re


# CLEAN REPETITIONS

def fix_common_errors(text):

    # remove repeated words
    text = re.sub(
        r'\b(\w+)( \1){2,}\b',
        r'\1',
        text
    )

    # remove repeated 2-word phrases
    text = re.sub(
        r'\b(\w+ \w+)( \1)+\b',
        r'\1',
        text
    )

    # remove repeated 3-word phrases
    text = re.sub(
        r'\b(\w+ \w+ \w+)( \1)+\b',
        r'\1',
        text
    )

    return text


# BEAM SEARCH GENERATION

def beam_search_generate(
    model,
    features,
    idx2word,
    beam_width=10,
    max_len=50,
    alpha=0.8
):

    model.eval()

    features = features.unsqueeze(0).to(device)

    with torch.no_grad():

        memory = model.encode(features)

        beams = [
            ([SOS_IDX], 0.0)
        ]

        completed = []

        for _ in range(max_len):

            new_beams = []

            for tokens, score in beams:

                if tokens[-1] == EOS_IDX:

                    completed.append(
                        (tokens, score)
                    )

                    continue

                tgt = torch.tensor(
                    [tokens],
                    dtype=torch.long
                ).to(device)

                tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                    tgt.shape[1]
                ).to(device)

                output = model.decode(
                    tgt,
                    memory,
                    tgt_mask=tgt_mask
                )

                logits = output[:, -1, :]

                probs = F.log_softmax(
                    logits,
                    dim=-1
                )

                # repetition penalty
                repetition_penalty = 1.3

                for token_id in set(tokens):

                    if token_id not in [
                        SOS_IDX,
                        EOS_IDX,
                        BLANK_IDX
                    ]:

                        probs[0][token_id] /= repetition_penalty

                topk = torch.topk(
                    probs,
                    beam_width
                )

                for i in range(beam_width):

                    next_token = topk.indices[0][i].item()

                    next_score = (
                        score
                        + topk.values[0][i].item()
                    )

                    new_beams.append(
                        (
                            tokens + [next_token],
                            next_score
                        )
                    )

            beams = sorted(
                new_beams,
                key=lambda x: x[1],
                reverse=True
            )[:beam_width]

        completed.extend(beams)

        truly_completed = [

            b for b in completed

            if b[0][-1] == EOS_IDX
        ]

        if not truly_completed:

            truly_completed = completed

        best_tokens = sorted(

            truly_completed,

            key=lambda x: (
                x[1]
                / (len(x[0]) ** alpha)
            ),

            reverse=True

        )[0][0]

        words = []

        for idx in best_tokens:

            if idx in [
                SOS_IDX,
                EOS_IDX,
                BLANK_IDX
            ]:
                continue

            if idx in idx2word:

                words.append(
                    idx2word[idx]
                )

        sentence = " ".join(words)

        sentence = fix_common_errors(
            sentence
        )

        return sentence


# RANDOM VALIDATION SAMPLE

idx = random.randint(
    0,
    len(val_dataset) - 1
)

features, actual_sentence = val_dataset[idx]

predicted_sentence = beam_search_generate(
    model,
    features,
    idx2word,
    beam_width=5
)

print("\n========== RESULT ==========")

print("\nACTUAL:")
print(actual_sentence)

print("\nPREDICTED:")
print(predicted_sentence)


========== RESULT ==========

ACTUAL:
am samstag in der südwesthälfte meist trocken sonst mal sonne mal schauer im nordosten gewitter

PREDICTED:
am samstag ist es im südwesten meist trocken im südwesten einzelne schauer und gewitter
